# EDA e Treino — LSTM PETR4

Exploração dos dados e validação visual do modelo. O treino de produção que
gera os artefatos está em `src/train_pipeline.py`; aqui o foco são os gráficos
que justificam as decisões de modelagem.

Etapas:
1. Histórico de PETR4 via yfinance (2015 em diante)
2. Split temporal 80/10/10 (sem embaralhar)
3. MinMaxScaler ajustado só no treino (evita leak)
4. Janelas de 60 dias prevendo o próximo fechamento
5. LSTM(64) -> LSTM(32) com Dropout, EarlyStopping e ReduceLROnPlateau
6. Avaliação com MAE, RMSE e MAPE em escala original

## 0. Setup (Colab **ou** local)

A célula abaixo detecta o ambiente automaticamente:
- **No Colab:** clona o repositório e instala as dependências que faltam (yfinance, curl_cffi, pydantic-settings). O resto (TensorFlow, pandas, sklearn, matplotlib) já vem no Colab.
- **Local:** apenas adiciona a raiz do projeto ao `sys.path`.

Sem isso, no Colab você toma `ModuleNotFoundError: No module named 'src'`, porque o código-fonte (`src/`) não estaria presente.

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = '/content/tech_challenge_4'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', '-q',
             'https://github.com/gbsander/tech_challenge_4.git', REPO],
            check=True,
        )
    # deps que o Colab nao traz por padrao
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'yfinance==1.3.0', 'curl_cffi', 'pydantic-settings'],
        check=True,
    )
else:
    # local: sobe a partir do cwd ate achar a pasta que contem src/
    REPO = os.path.abspath('.')
    while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, 'src')):
        REPO = os.path.dirname(REPO)

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('Ambiente Colab:', IN_COLAB)
print('Raiz do projeto:', REPO)
assert os.path.isdir(os.path.join(REPO, 'src')), f'src/ nao encontrado em {REPO}'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import build_dataset
from src.model import build_model, train
from src.evaluate import compute_metrics

TICKER = 'PETR4.SA'
WINDOW = 60
START = '2015-01-01'

## 1. Coleta + EDA

In [ ]:
ds = build_dataset(
    ticker=TICKER, start_date=START, end_date=None,
    window=WINDOW, train_frac=0.8, val_frac=0.1,
)
close = ds.raw_close
print(f'Pontos: {len(close)}, range: {close.index.min()} → {close.index.max()}')
print(f'Min={close.min():.2f}  Max={close.max():.2f}  Média={close.mean():.2f}  Std={close.std():.2f}')
close.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(close.index, close.values, lw=0.8, color='#1f77b4')
n = len(close)
split_train = close.index[int(n * 0.8)]
split_val = close.index[int(n * 0.9)]
ax.axvline(split_train, color='orange', ls='--', label='train→val')
ax.axvline(split_val, color='red', ls='--', label='val→test')
ax.set_title(f'{TICKER} — Close (2015 → hoje) com split temporal')
ax.set_ylabel('R$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Retornos diários
returns = close.pct_change().dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(returns.index, returns.values, lw=0.5, color='gray')
axes[0].set_title('Retorno diário')
axes[1].hist(returns.values, bins=80, color='#2ca02c', alpha=0.7)
axes[1].set_title('Distribuição dos retornos')
plt.tight_layout()
plt.show()
print(f'Vol diária: {returns.std():.4f}  →  anualizada ~{returns.std() * (252 ** 0.5):.4f}')

## 2. Modelo + treino

In [ ]:
model = build_model(window=WINDOW, n_features=1, learning_rate=1e-3)
model.summary()

In [ ]:
history = train(
    model,
    ds.X_train, ds.y_train,
    ds.X_val, ds.y_val,
    epochs=50, batch_size=32,
)

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h['loss'], label='train')
axes[0].plot(h['val_loss'], label='val')
axes[0].set_title('Loss (MSE) por época'); axes[0].legend()
axes[1].plot(h['mae'], label='train')
axes[1].plot(h['val_mae'], label='val')
axes[1].set_title('MAE por época'); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Avaliação no test set

In [ ]:
test_pred = ds.scaler.inverse_transform(model.predict(ds.X_test, verbose=0)).ravel()
test_true = ds.scaler.inverse_transform(ds.y_test).ravel()
metrics = compute_metrics(test_true, test_pred)
print(f"MAE  = {metrics['mae']:.4f}  (R$)")
print(f"RMSE = {metrics['rmse']:.4f}  (R$)")
print(f"MAPE = {metrics['mape']:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_true, label='Real', lw=1.0)
ax.plot(test_pred, label='Predito', lw=1.0, alpha=0.8)
ax.set_title(f'{TICKER} — Real vs Predito (test set)')
ax.set_xlabel('Pregão (índice)'); ax.set_ylabel('R$'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Resíduos
resid = test_true - test_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(resid, lw=0.7); axes[0].axhline(0, color='red', ls='--')
axes[0].set_title('Resíduos (real - predito)')
axes[1].hist(resid, bins=40, color='#d62728', alpha=0.7)
axes[1].set_title('Distribuição dos resíduos')
plt.tight_layout(); plt.show()
print(f'Mean resid: {resid.mean():.4f}  Std: {resid.std():.4f}')

## 4. Forecast multi-step (autoregressivo)

Demo do que o endpoint `/predict/forecast?horizon=N` faz: usa a predição do passo t como input do passo t+1.

In [ ]:
horizon = 10
last_window = close.values[-WINDOW:]
scaled = ds.scaler.transform(last_window.reshape(-1, 1)).ravel()
window = scaled.copy()
preds_scaled = []
for _ in range(horizon):
    x = window.reshape(1, WINDOW, 1).astype(np.float32)
    yhat = float(model.predict(x, verbose=0).ravel()[0])
    preds_scaled.append(yhat)
    window = np.concatenate([window[1:], [yhat]])
preds = ds.scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).ravel()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(WINDOW), last_window, label='últimos 60 (real)')
ax.plot(range(WINDOW, WINDOW + horizon), preds, label=f'forecast ({horizon}d)', color='red', marker='o')
ax.set_title(f'{TICKER} — forecast autoregressivo')
ax.legend(); plt.tight_layout(); plt.show()
print('Predições:', [f'{p:.2f}' for p in preds])

## Notas

- **Limitação do forecast multi-step**: erro acumula por causa da realimentação. Bom pra horizontes curtos (≤ 10 dias).
- **Por que univariate**: foco no challenge (LSTM básico funciona bem). Multivariate (OHLCV + indicadores técnicos) está como próximo passo no README.
- **Por que MinMaxScaler só no treino**: caso fosse fittado em todo o histórico, o modelo teria visto info do futuro durante treino (data leakage).